# In-Context Learning


In-context learning is a generalisation of few-shot learning where the LLM is provided a context as part of the prompt and asked to respond by utilising the information in the context.

* Example: *"Summarize this research article into one paragraph highlighting its strengths and weaknesses: [insert article text]”*
* Example: *"Extract all the quotes from this text and organize them in alphabetical order: [insert text]”*

A very popular technique that you will learn in week 5 called Retrieval-Augmented Generation (RAG) is a form of in-context learning, where:
* a search engine is used to retrieve some relevant information
* that information is then provided to the LLM as context


In this example we download some recent research papers from arXiv papers, extract the text from the PDF files and ask Gemini to summarize the articles as well as provide the main strengths and weaknesses of the papers. Finally we print the summaries to a local html file and as markdown.

In [ ]:
!pip install requests bs4 google-generativeai pypdf

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
import google.generativeai as genai
from urllib.request import urlopen, urlretrieve
from IPython.display import Markdown, display
from pypdf import PdfReader
from datetime import date
from tqdm import tqdm
from google.colab import userdata

In [ ]:
#API_KEY = os.environ.get("GEMINI_API_KEY")
API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=API_KEY)

We select those papers that have been featured in Hugging Face papers.

In [ ]:
BASE_URL = "https://huggingface.co/papers"
page = requests.get(BASE_URL)
soup = BeautifulSoup(page.content, "html.parser")
h3s = soup.find_all("h3")

papers = []

for h3 in h3s:
    a = h3.find("a")
    title = a.text
    link = a["href"].replace('/papers', '')

    papers.append({"title": title, "url": f"https://arxiv.org/pdf{link}"})

Code to extract text from PDFs.

In [ ]:
def extract_paper(url):
    html = urlopen(url).read()
    soup = BeautifulSoup(html, features="html.parser")

    # kill all script and style elements
    for script in soup(["script", "style"]):
        script.extract()    # rip it out

    # get text
    text = soup.get_text()

    # break into lines and remove leading and trailing space on each
    lines = (line.strip() for line in text.splitlines())
    # break multi-headlines into a line each
    chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
    # drop blank lines
    text = '\n'.join(chunk for chunk in chunks if chunk)

    return text


def extract_pdf(url):
    pdf = urlretrieve(url, "pdf_file.pdf")
    reader = PdfReader("pdf_file.pdf")
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text


def printmd(string):
    display(Markdown(string))

In [ ]:
LLM = "gemini-2.5-flash"
model = genai.GenerativeModel(LLM)

We use Gemini to summarize the papers.

In [ ]:
for paper in tqdm(papers):
    try:
        paper["summary"] = model.generate_content("Summarize this research article into one paragraph by highlighting its strengths and weaknesses. Format it into a table where strengths and weaknessess are in a different columns" + extract_pdf(paper["url"])).text
    except:
        print("Generation failed")
        paper["summary"] = "Paper not available"

 80%|████████  | 12/15 [20:05<13:14, 264.75s/it]

Generation failed


 87%|████████▋ | 13/15 [30:07<12:13, 366.83s/it]

Generation failed


 93%|█████████▎| 14/15 [40:08<07:17, 437.67s/it]

Generation failed


100%|██████████| 15/15 [50:10<00:00, 200.69s/it]

Generation failed


We print the results to a html file.

In [ ]:
page = f"<html> <head> <h1>Daily Dose of AI Research</h1> <h4>{date.today()}</h4> <p><i>Summaries generated with: {LLM}</i>"
with open("papers.html", "w") as f:
    f.write(page)
for paper in papers:
    page = f'<h2><a href="{paper["url"]}">{paper["title"]}</a></h2> <p>{paper["summary"]}</p>'
    with open("papers.html", "a") as f:
        f.write(page)
end = "</head>  </html>"
with open("papers.html", "a") as f:
    f.write(end)

We can also print the results to this notebook as markdown.

In [ ]:
for paper in papers:
    printmd("**[{}]({})**<br>{}<br><br>".format(paper["title"],
                                                paper["url"],
                                                paper["summary"]))

**[Thinking with Video: Video Generation as a Promising Multimodal
  Reasoning Paradigm](https://arxiv.org/pdf/2511.04570)**<br>This research introduces "Thinking with Video," a novel multimodal reasoning paradigm leveraging video generation models like Sora-2. The study's strengths lie in demonstrating Sora-2's robust performance on vision-centric tasks, where it often matches or surpasses state-of-the-art VLMs in spatial and inductive reasoning, and its surprising capability on diverse text-centric benchmarks, supported by the new VideoThinkBench. It also highlights Sora-2's few-shot learning and the performance-boosting potential of self-consistency in video generation. However, key weaknesses include Sora-2's difficulty in visually articulating coherent reasoning steps, the strong indication that its text-centric prowess derives from an internal prompt rewriter rather than the video generation process itself, its comparative struggle on more complex reasoning problems, and the inherent limitation of evaluating a closed-source model.

| Strengths                                                                                                                                                                                                                                                                                                                                                      | Weaknesses                                                                                                                                                                                                                                                                                                                                                     |
| :------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | :------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Novel Paradigm:** Proposes "Thinking with Video" as a promising unified multimodal reasoning paradigm, overcoming limitations of text/image-only approaches.                                                                                                                                                                                                    | **Poor Visual Reasoning Process Generation:** Sora-2 struggles to generate coherent and readable reasoning processes visually within video frames, even when the final answer is correct.                                                                                                                                                                          |
| **Strong Vision-Centric Performance:** Sora-2 demonstrates robust spatial and inductive reasoning on vision-centric tasks (e.g., Eyeballing Puzzles), often comparable to or surpassing state-of-the-art Vision-Language Models (VLMs).                                                                                                                            | **Prompt Rewriter Dependency:** The impressive text-centric reasoning is strongly hypothesized to originate from an internal prompt rewriter model (like in Wan2.5), rather than the video generation model's inherent reasoning capabilities for text.                                                                                                            |
| **Surprising Text-Centric Capability:** Achieves high accuracy on various text-centric benchmarks (e.g., 98.9% on GSM8K, 94.0% on MATH, 81.1% on MathVista) by embedding text within video frames.                                                                                                                                                                | **Lags on Challenging Text Tasks:** Sora-2's performance significantly lags behind SOTA VLMs on more complex text-centric datasets such as AIME, GPQA, and MMMU.                                                                                                                                                                                          |
| **Comprehensive Benchmark:** Introduces VideoThinkBench, a systematically constructed benchmark encompassing diverse vision-centric (verifiable) and adapted text-centric tasks for evaluating video generation models.                                                                                                                                              | **Limited Performance on Complex Vision Tasks:** Sora-2 struggles with more complex maze geometries (e.g., hexagon/circle mazes) and does not strictly match dataset annotations for abstract reasoning tasks like ARC-AGI-2, despite making "reasonable predictions."                                                                                              |
| **Few-Shot Learning:** Demonstrates that Sora-2 is a few-shot learner, with performance on abstract reasoning tasks (ARC-AGI-2) improving when provided with more examples.                                                                                                                                                                                        | **Closed-Source Model Limitation:** The primary model evaluated (Sora-2) is proprietary and closed-source, which fundamentally limits the ability for in-depth internal analysis, understanding its mechanisms, and reproducibility of results for the research community.                                                                                            |
| **Self-Consistency for Improvement:** Shows that self-consistency can significantly boost Sora-2's performance in verifiable video generation reasoning tasks, opening an underexplored direction for test-time scaling.                                                                                                                                             | **Subjective Evaluation for Visual Puzzles:** Sora-2's evaluation on visual puzzles relied on manual assessment of "best" video frames, which could introduce some level of subjectivity compared to automated evaluations.                                                                                                                                  |
| **Systematic Comparison:** Provides a systematic comparison of video generation models (Sora-2) against SOTA VLMs, highlighting the advantages of "Thinking with Video."                                                                                                                                                                                          | **Narrow Model Scope:** The evaluation primarily focuses on Sora-2, with limited inclusion of other video generation models, especially open-source alternatives, which would provide a broader understanding of the paradigm's generalizability.                                                                                                         |
| **Test Set Leakage Analysis:** Conducted experiments with derived math problems (with different numerical values) to effectively reduce the likelihood of test set leakage for text-centric tasks, validating Sora-2's inherent reasoning potential. | **Potential for Misinterpretation:** The observed "multimodal understanding and generation" for text-centric tasks might be more a function of the prompt rewriter's textual processing than the video model's direct comprehension of the problem through video. |<br><br>

**[V-Thinker: Interactive Thinking with Images](https://arxiv.org/pdf/2511.04460)**<br>This research introduces V-Thinker, a novel general-purpose multimodal reasoning assistant designed to empower Large Multimodal Models (LMMs) with interactive, vision-centric thinking through an end-to-end reinforcement learning framework. Its key strengths include an innovative Data Evolution Flywheel that automatically synthesizes, evolves, and verifies diverse, high-quality, and progressively difficult interactive reasoning datasets, along with a Visual Progressive Training Curriculum for aligning perception and interactive reasoning capabilities. The paper further contributes VTBench, an expert-verified benchmark specifically designed for tasks inherently requiring visual interaction. While V-Thinker consistently outperforms strong LMM baselines across both interactive (on VTBench) and general reasoning scenarios, demonstrating robust interactive thinking capabilities, a notable weakness stems from its current limited model scale due to computational constraints, which reduces its effectiveness in highly knowledge-intensive or specialized domain tasks, as explicitly acknowledged by the authors.

Here's the summary formatted into a table:

| Strengths                                                                                                                                                                                                                                                                                                                                                              | Weaknesses                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| :--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | :--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Novel Framework:** Introduces V-Thinker, a general-purpose multimodal reasoning assistant for interactive, vision-centric thinking via end-to-end reinforcement learning, addressing a critical gap in LMMs.                                                                                                                                                            | **Limited Model Scale:** Due to computational constraints, the current model iteration has a limited scale, which restricts its capability in knowledge-intensive or highly domain-specific tasks.                                                                                                                                                                                                                                                           |
| **Innovative Data Synthesis (Data Evolution Flywheel):** Automatically synthesizes, evolves, and verifies diverse, high-quality, and difficulty-stratified interactive reasoning datasets (V-Interaction-400K, V-Perception-40K) through a self-sustaining knowledge-driven co-evolution mechanism.                                                                    | **Reliance on Closed-Source Models for Data Generation:** The data evolution flywheel explicitly uses GPT-5 as the "strong generator G" for creating initial QA pairs, code, and images, introducing a dependency on a closed-source, powerful model for a core part of their data generation process.                                                                                                                                                                 |
| **Comprehensive Training Curriculum:** Features a Visual Progressive Training Curriculum that effectively aligns perception via point-level supervision and then integrates interactive reasoning through a two-stage (Supervised Fine-Tuning + Reinforcement Learning) framework.                                                                                     | **Potential for Noise/Inaccuracy in Image-to-Code:** The paper acknowledges that "a heavy reliance on 'Image2Code' pipelines for image editing struggles to accurately depict spatial relationships between visual elements and may introduce extra noise" (Introduction). While V-Thinker uses code-driven tools, the inherent challenge of perfect visual code generation remains a subtle weakness.                                                                   |
| **New Expert-Verified Benchmark (VTBench):** Introduces a specialized benchmark for vision-centric interactive reasoning tasks, which is expert-verified and covers hierarchical evaluation dimensions (Perception, Instruction-Guided Interaction, Interactive Reasoning).                                                                                                  |                                                                                                                                                                                                                                                                                                                                                                                                                                                  |
| **Strong Empirical Performance:** Consistently outperforms mainstream LMM-based baselines across both interactive (VTBench) and general reasoning benchmarks (MathVision, We-Math, VisuLogic), demonstrating superior interactive thinking and generalization capabilities.                                                                                             |                                                                                                                                                                                                                                                                                                                                                                                                                                                  |
| **Qualitative Evidence:** Provides compelling qualitative analysis, including visual interaction editing, multi-dimensional derivative reasoning in rollout sampling, and complete interactive reasoning trajectories, showcasing human-like problem-solving.                                                                                                            |                                                                                                                                                                                                                                                                                                                                                                                                                                                  |<br><br>

**[Scaling Agent Learning via Experience Synthesis](https://arxiv.org/pdf/2511.03773)**<br>DreamGym introduces a novel framework to overcome the challenges of scaling Reinforcement Learning (RL) for Large Language Model (LLM) agents, such as high interaction costs, limited task diversity, and unstable rewards. It achieves this by synthesizing diverse experiences through a reasoning-based experience model that abstractly simulates environment dynamics, an experience replay buffer initialized with real data, and an adaptive curriculum task generator that creates progressively challenging scenarios. This approach significantly boosts RL training efficiency and performance, particularly in non-RL-ready environments like WebArena where it outperforms baselines by over 30%, and also matches or surpasses traditional RL in costly settings with purely synthetic data or via efficient sim-to-real transfer. While DreamGym excels in providing scalable, stable, and generalizable training within its abstract meta-representation space, its cross-domain transferability is limited when faced with large domain gaps, and its current scope is primarily focused on single-environment learning setups rather than a universal world model.

| Strengths                                                                                                                                                                                                                                                                                                                                                        | Weaknesses                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| :--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | :---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Scalability & Cost-Effectiveness:** Dramatically reduces the high cost and complexity of real-environment rollouts by synthesizing diverse, reasoning-driven experiences, making RL training more accessible.                                                                                                                                                          | **Limited Cross-Domain Transferability:** While effective for similar domains, policy performance drops significantly when transferring between vastly different environments (e.g., web-based to embodied control), indicating current limitations in the universality of its meta-representations.                                                                                                                                                                                                                                   |
| **Enables RL in Non-RL-Ready Environments:** Makes RL training feasible and highly effective in domains like WebArena that traditionally lack scalable data collection, reliable rewards, or environment reset mechanisms.                                                                                                                                            | **Single-Environment Focus:** The framework primarily investigates single-environment learning setups, and its current design does not yet unify multiple environment models for seamless knowledge transfer across fundamentally distinct domains.                                                                                                                                                                                                                                                                                           |
| **High Performance & Sample Efficiency:** Achieves substantial performance gains (e.g., >30% on WebArena), matches traditional RL baselines with purely synthetic data, and provides an efficient warm-start strategy for sim-to-real transfer with significant improvements using minimal real-world interactions.                                                | **Dependence on LLM Reasoning Quality:** The core of DreamGym relies on an LLM-based reasoning experience model, whose effectiveness and fidelity in generating consistent and informative state transitions are inherently tied to the capabilities and careful prompting of the underlying LLM, potentially introducing fragility or reliance on powerful external models for initial training data annotation.                                                                                                                                 |
| **Stable & Quality Training:** Generates dense and consistent feedback through curriculum-driven rollout synthesis, leading to smoother learning curves and mitigating the training instabilities commonly observed in real environments.                                                                                                                                |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 |
| **Generalization & Behavioral Priors:** Policies trained in DreamGym demonstrate strong generalization and cross-domain transferability within abstract meta-representation spaces, enabling agents to learn domain-agnostic behavioral priors rather than task-specific patterns.                                                                                |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 |<br><br>

**[Cambrian-S: Towards Spatial Supersensing in Video](https://arxiv.org/pdf/2511.04670)**<br>This research article introduces "spatial supersensing" as a critical advancement for multimodal intelligence, urging MLLMs to evolve beyond linguistic-only understanding towards implicit 3D world modeling and prediction from continuous video streams. To this end, the authors present VSI-SUPER, a novel benchmark for long-horizon spatial recall and continual counting, which effectively exposes the severe limitations of current state-of-the-art MLLMs like Gemini-2.5-Flash in handling unbounded context and achieving true spatial generalization. To address these gaps, they curate VSI-590K, a large-scale, spatially-focused instruction-tuning dataset, and train Cambrian-S, a family of spatially-grounded MLLMs that achieves state-of-the-art performance on existing spatial cognition benchmarks with robust generalization. Recognizing that data scaling alone is insufficient, the paper proposes "predictive sensing," a proof-of-concept leveraging self-supervised latent frame prediction and "surprise" (prediction error) for efficient memory management and event segmentation, which substantially outperforms proprietary baselines on VSI-SUPER by enabling models to actively anticipate and organize visual experience.

| Strengths                                                                                                                                                                                                                                                              | Weaknesses                                                                                                                                                                                                                                                        |
| :--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | :---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Conceptual Clarity:** Proposes a clear hierarchy for "spatial supersensing," guiding future research in multimodal intelligence beyond linguistic-only understanding.                                                                                                  | **Early Stage Prototype:** The "predictive sensing" solution is presented as an "early prototype" and "proof of concept," acknowledging current limitations in quality, scale, and generalizability.                                                                    |
| **Novel & Challenging Benchmark (VSI-SUPER):** Introduces VSI-SUPER (VSR for recall, VSC for counting) which effectively exposes the limitations of current MLLMs in continuous, long-horizon spatial reasoning and is resistant to brute-force context expansion.     | **Limited Scope of Supersensing Achieved:** The paper explicitly states that it does not "realize supersensing here," indicating that the presented solution is an initial step rather than a comprehensive achievement of the full conceptual hierarchy.               |
| **High-Quality, Spatially-Focused Dataset (VSI-590K):** Curates a large, diverse dataset (real, simulated, pseudo-annotated videos) with systematic question types, addressing a critical data gap for visual-spatial understanding.                                     | **Benchmark Generalizability:** While challenging, VSI-SUPER, built from edited/concatenated room-tour videos, might have limitations in fully representing the diverse and complex nature of all "unbounded visual streams" in real-world scenarios.                     |
| **State-of-the-Art Model (Cambrian-S):** Develops Cambrian-S, a robust spatially-grounded MLLM family that achieves SOTA on existing spatial cognition benchmarks, generalizes well to unseen spatial tasks, and maintains competitive general video understanding.        | **Persistent Challenge of Data/Model Scaling:** Acknowledges that even with significant data and engineering effort (Cambrian-S), purely data-driven approaches within the current MLLM framework still face "fundamental limits" in achieving true spatial supersensing. |
| **Innovative "Predictive Sensing" Paradigm:** Proposes and prototypes a new direction leveraging self-supervised latent frame prediction and "surprise" (prediction error) for efficient memory management and event segmentation, offering a principled approach. | **Need for Further Research:** Explicitly calls for future work to explore more diverse and embodied scenarios and to build stronger connections with recent advances in vision, language, and world modeling, highlighting the nascent stage of this research.         |<br><br>

**[NVIDIA Nemotron Nano V2 VL](https://arxiv.org/pdf/2511.03929)**<br>NVIDIA's Nemotron Nano V2 VL is an advanced 12B vision-language model, showcasing significant improvements over its predecessor across diverse tasks like real-world document understanding, long video comprehension, and reasoning. Its strengths lie in its hybrid Mamba-Transformer architecture enabling high throughput for long contexts and videos via techniques like Efficient Video Sampling (EVS), comprehensive multi-stage training, and impressive open-source commitment through releasing models (BF16, FP8, FP4), datasets, recipes, and tools. A notable weakness, however, was the initial degradation of pure-text reasoning capabilities (e.g., LiveCodeBench, RULER) after multimodal fine-tuning, necessitating dedicated recovery stages. Additionally, while the model is generally robust, alternative image processing strategies presented a persistent performance gap on specific OCR benchmarks, indicating areas for further refinement in visual processing.

| Strengths                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       | Weaknesses                                                                                                                                                                                                                                                                                                                                                                                                                                                        |
| :----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | :-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Performance:** Achieves significant improvements over its predecessor (Llama-3.1-Nemotron-Nano-VL-8B) across all vision and text domains. Delivers leading accuracy on OCRBench v2 and strong performance in complex tasks like real-world document understanding, long video comprehension, and STEM reasoning.                                                                                                                                                                                                                                                                                                                                                                                                                      | **Initial Text Reasoning Degradation:** Multimodal fine-tuning (SFT Stage 1) led to a "substantial drop" in pure-text reasoning benchmarks (e.g., LiveCodeBench, RULER), requiring dedicated "text recovery" stages (SFT Stages 3 and 4) to restore these capabilities, suggesting a challenge in seamlessly integrating multimodal and text-only training.                                                                                                    |
| **Efficiency & Scalability:** Incorporates a hybrid Mamba-Transformer LLM backbone, innovative token reduction techniques, and Efficient Video Sampling (EVS) to achieve high inference throughput (35% higher for long documents, 2x+ for video) with minimal accuracy loss. Extends context length from 16K to 128K (and up to 300K in training stages). Supports FP8 and FP4 quantization for further efficiency.                                                                                                                                                                                                                                                                                                              | **Specific Visual Processing Gaps:** While generally robust, alternative image processing strategies (e.g., native resolution without tiling) showed a "notable performance drop on OCRBench and OCRBench-V2 (English)," with the gap on OCRBench-V2 (English) persisting even with tiling-size matching, indicating room for improvement in handling certain image input modalities or specific OCR challenges. |
| **Robust Training Methodology:** Employs a comprehensive multi-stage training approach (warm-up, SFT with context extension and dedicated code reasoning recovery) that effectively balances and enhances both multimodal and pure-text capabilities. Includes both reasoning-on and reasoning-off modes, with tunable reasoning budget control shown to improve accuracy.                                                                                                                                                                                                                                                                                                                                                         | **Minor Quantization Accuracy Drop:** Post-Training Quantization (PTQ) to NVFP4 initially resulted in a "small accuracy drop," necessitating the additional step of Quantization-Aware Distillation (QAD) to recover quality, highlighting a minor challenge in achieving full fidelity with extreme quantization without further optimization.                                                                                        |
| **Open-Source Contribution:** Commits to open-sourcing by releasing model weights in multiple precision formats (BF16, FP8, FP4), significant portions of their extensive training datasets (Nemotron VLM Dataset V2), training recipes, and tooling (NVPDFTex), fostering community research and development.                                                                                                                                                                                                                                                                                                                                                                                                                                  |                                                                                                                                                                                                                                                                                                                                                                                                                                                                 |
| **Comprehensive Evaluation:** Conducts extensive evaluations across 45 benchmarks spanning seven broad categories (General VQA, STEM Reasoning, Document Understanding, Visual Grounding, GUI Understanding, Video Understanding, and Multimodal Multilingual Understanding), providing a thorough assessment of the model's capabilities and competitive comparisons against state-of-the-art models of similar scale. |                                                                                                                                                                                                                                                                                                                                                                                                                                                                 |<br><br>

...

**[Learning Vision-Driven Reactive Soccer Skills for Humanoid Robots](https://arxiv.org/pdf/2511.03996)**<br>This research presents a novel unified reinforcement learning (RL) controller that enables humanoid robots to acquire reactive soccer skills by directly integrating visual perception and motion control. Building on Adversarial Motion Priors (AMP), the approach employs an encoder-decoder architecture alongside a virtual perception system to robustly handle real-world visual characteristics and imperfect observations, facilitating active perception-action coordination. The controller demonstrates exceptional reactivity and robustness, achieving zero-shot sim-to-real transfer across diverse environments and winning the RoboCup 2025 Adult-size Humanoid League championship, outperforming traditional rule-based methods in speed and agility. However, its current scope is limited to individual soccer skills, lacking mechanisms for multi-robot coordination and broader human-like behaviors such as dribbling or strategic passing, with the goal-oriented training potentially restricting behavioral diversity.

| Strengths                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  ```
| **Strengths**                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       |
| --- | --- |
| - **Unified RL Controller:** Directly integrates visual perception and motion control to achieve advanced, reactive soccer behaviors, circumventing the limitations of traditional decoupled systems.                                                                                                                                     | - **Lack of Multi-Robot Context:** The current system primarily focuses on individual skill execution, with observations limited to the ball and goal. It lacks mechanisms for perceiving and responding to opponents or coordinating with teammates, which is crucial for complex team strategies. |
| - **AMP Extended for Perception:** Successfully extends Adversarial Motion Priors (AMP) to real-world dynamic environments involving visual feedback, effectively bridging motion imitation with visually grounded dynamic control.                                                                                                          | - **Limited Skill Repertoire Compared to Humans:** While demonstrating impressive walking and adaptive kicking, the policy does not yet encompass the full range of human soccer skills such as dribbling, ball trapping, or strategic passing.                                      |
| - **Robust Perception-Action Coordination:** Achieved through an encoder-decoder network and a virtual perception system that models real-world visual characteristics. This design enables the policy to denoise imperfect observations, accurately estimate ball positions, and actively coordinate its movements (e.g., head and torso orientation) with visual input. | - **Potential for Restricted Behavioral Diversity:** The goal-oriented nature of the current training framework, while effective for achieving task success, can lead the policy to converge on a limited set of movement patterns, potentially hindering the development of more varied and context-aware behaviors. |
| - **Exceptional Reactivity and Coherence:** The controller demonstrates strong reactivity, consistently executing coherent and robust soccer behaviors like ball searching, chasing, and multi-directional kicking, all seamlessly integrated without manual skill segmentation.                                          | |
| - **Zero-Shot Sim-to-Real Transfer:** The policy, trained entirely in simulation, proved highly effective and robust when deployed directly onto physical robots across diverse real-world terrains and challenging visual conditions.                                                                                          | |
| - **Validated in Competitive Environments:** The method's practical efficacy was confirmed by the Tsinghua Hephaestus team winning championships in the RoboCup 2025 Adult-size Humanoid League and the 2025 World Humanoid Robot Games under strict real-world constraints.                                          | |
| - **Superior Performance against Rule-Based Systems:** The learned policy consistently outperformed state-of-the-art rule-based strategies in terms of kicking time and agility, demonstrating a more fluid and integrated approach from ball approach to kick execution.                                                | |
| - **Adaptive and Versatile Gait:** The controller exhibits a versatile set of gaits, including adaptive step frequencies and agile maneuvers (e.g., rapid turns, pivot hook kicks) tailored to ball proximity and movement, optimizing both efficiency and precision.                                                 | |
| - **Efficient and Stable Training:** Utilizes a single-stage training process, a multi-critic framework to reduce interference between reward components, and GAN-based adversarial learning for implicit motion guidance.                                                                                        | |
| - **Simplified Architecture and Robustness:** A unified policy mapping compact visual detections directly to motor commands simplifies the system and enhances generalization. The integrated odometry module further improves robustness against temporary visual information loss. | |<br><br>

**[How to Evaluate Speech Translation with Source-Aware Neural MT Metrics](https://arxiv.org/pdf/2511.03295)**<br>Paper not available<br><br>

**[RDMA Point-to-Point Communication for LLM Systems](https://arxiv.org/pdf/2510.27656)**<br>Paper not available<br><br>

**[SAIL-RL: Guiding MLLMs in When and How to Think via Dual-Reward RL
  Tuning](https://arxiv.org/pdf/2511.02280)**<br>Paper not available<br><br>

**[EVTAR: End-to-End Try on with Additional Unpaired Visual Reference](https://arxiv.org/pdf/2511.00956)**<br>Paper not available<br><br>